In [15]:
"""
Updated on Fri Oct 31 
@author: Jingyi 
Created on Frid Jan 21 10:02:01 2022
@author: Oumbeg
"""
from bs4 import BeautifulSoup
import datetime
import pandas as pd
import numpy as np
from pandas import ExcelWriter
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from time import sleep
import os
import re
import requests
import tabula
import camelot
from docx2pdf import convert

print("Running UG BUG Web Scraping Tool v.1.0")
regulatorName = 'UG BUG'

#Assigning current time, output file name and ExcelWriter object
now = datetime.datetime.now()
filename = 'UG BUG SQL Ready {}.xlsx'.format(str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#Assigning the folders that are going to be used in the process
#scriptfolder = os.path.dirname(os.path.abspath(__file__))
tempfolder = os.path.join(scriptfolder,'tempfolder')
os.chdir(scriptfolder)

#Creating tempfolder if it doesn't exists, emptying in if it does exist
if os.path.exists(tempfolder):
	for temp_file in os.listdir(tempfolder):
		os.remove(os.path.join(tempfolder, temp_file))
else:
	os.mkdir(tempfolder)

regdict={'UG BUG 1': 'https://bou.or.ug/bouwebsite/Supervision/supervisedinstitutions.html'}

chromeOptions = webdriver.ChromeOptions()
prefs = {"download.default_directory" : tempfolder, 
        "plugins.always_open_pdf_externally": True}
chromeOptions.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()


sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}


def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty

    return sqldict

def prepare_tempfolder(path):
    os.makedirs(path, exist_ok=True)

def wait_for_new_download(path, known):
    while True:
        files = [f for f in os.listdir(path)
                 if not f.endswith('.crdownload') and not f.endswith('.tmp')]
        new_files = [f for f in files if f not in known]
        if new_files:
            return new_files[0]
        print('Waiting for file to download')
        sleep(2)

# try to create an empty folder "tempfolder"
try:
    os.mkdir(tempfolder)
except:
    prevfiles=os.listdir(tempfolder)
    for prf in prevfiles:
        os.remove(os.path.join(tempfolder, prf))

processdate=now.strftime('%Y-%m-%d')

pattern = re.compile('([0-9]+)')

for reg in regdict:
    
    print('Working with {}'.format(reg))
    driver.get(regdict[reg])
    sleep(3)
    
    # the ListCode
    listCode = 0
    
    # the block where to find the links to download the PDFs 
    soup = BeautifulSoup(driver.page_source, "html.parser")
    table = soup.find("div",{"class":"contenttext"})
    typology = table.find_all("li")
    
    
    for m in range(len(typology)):
        d = m+1
        
        if d <= 5 and d not in [1,3,4]:
            driver.find_element(By.XPATH, '//*[@id="apollo-page"]/section/div/div[2]/div[2]/div[1]/div/div/div[2]/div/div/div/ul[1]/li['+ str(d) +']/a').click()
            while len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele])==0:
                print('Waiting for file to download')
                sleep(2)
    
            print(os.listdir(tempfolder))
            pdf_file = os.listdir(tempfolder)[0]
            filePath = os.path.join(tempfolder, pdf_file)
        
            print(m,'---',typology[m].text)
            if m == 0:
                if os.path.exists(tempfolder):
                    for temp_file in os.listdir(tempfolder):
                        os.remove(os.path.join(tempfolder, temp_file))
                else:
                    os.mkdir(tempfolder)
                continue

            sleep(3)
            tables = camelot.read_pdf(filePath, pages='all', flavor='lattice')
            # Iterate over the tables of each pages
            for i in range(tables.n):
                df_Table = tables[i].df
        
                for j in range(len(df_Table)):
                    if df_Table[1][j].strip() not in ['','NAME'] :
                        # print('Name: ',' '.join([item.strip() for item in df_Table[1][j].splitlines()]).strip())
                        # print('Address: ',' '.join([item.strip() for item in df_Table[2][j].splitlines()]).strip())
                        # print('TEL: ',' '.join([item.strip() for item in df_Table[3][j].splitlines()]).strip())
                        # print('FAX: ',' '.join([item.strip() for item in df_Table[4][j].splitlines()]).strip(),'\n')
                        sqldict['Name'].append(' '.join([item.strip() for item in df_Table[1][j].splitlines()]).strip())
                        sqldict['Address_1'].append(' '.join([item.strip() for item in df_Table[2][j].splitlines()]).strip())
                        sqldict['RegCtry'].append('UG')
                        sqldict['Cntry'].append('UG')
                        sqldict['Phone'].append(' '.join([item.strip() for item in df_Table[3][j].splitlines()]).replace('+','').replace('256','+256').strip())
                        if d ==2:

                            sqldict['BIC SWIFT Code'].append(' '.join([item.strip() for item in df_Table[4][j].splitlines()]).strip())
                        else:
                            if len(' '.join([item.strip() for item in df_Table[4][j].splitlines()]).strip())<3:
                                sqldict['Fax'].append('')
                            else:
                                sqldict['Fax'].append(' '.join([item.strip() for item in df_Table[4][j].splitlines()]).strip())
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['ListCode'].append(d)
                        sqldict['RegCode'].append('BUG')
                        sqldict['RegulationType'].append('Regulated')
                    sqldict = bourange_same_length_array(sqldict)
            os.remove(filePath)

                    

        elif d in [1]:
            print(m,'---',typology[m].text)
            driver.find_element(By.XPATH, '//*[@id="apollo-page"]/section/div/div[2]/div[2]/div[1]/div/div/div[2]/div/div/div/ul[1]/li['+ str(d) +']/a').click()
            while len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele])==0:
                print('Waiting for file to download')
                sleep(2)
            print(os.listdir(tempfolder))
            doc_file = os.listdir(tempfolder)[0]
            filePath = os.path.join(tempfolder, doc_file)   
            convert(filePath, "output.pdf")
            tables = camelot.read_pdf("output.pdf", pages='all', flavor='lattice')
            # Iterate over the tables of each pages
            for i in range(tables.n):
                df_Table = tables[i].df

                for j in range(len(df_Table)):
                    if df_Table[1][j].strip() not in ['','NAME'] :
                        # print('Name: ',' '.join([item.strip() for item in df_Table[1][j].splitlines()]).strip())
                        # print('Address: ',' '.join([item.strip() for item in df_Table[2][j].splitlines()]).strip())
                        # print('TEL: ',' '.join([item.strip() for item in df_Table[3][j].splitlines()]).strip())
                        # print('SWIFT: ',' '.join([item.strip() for item in df_Table[4][j].splitlines()]).strip(),'\n')
                        # print('E-MAIL AND WEBSITE: ',' '.join([item.strip() for item in df_Table[5][j].splitlines()]).strip(),'\n')
                        sqldict['Name'].append(' '.join([item.strip() for item in df_Table[1][j].splitlines()]).strip())
                        sqldict['Address_1'].append(' '.join([item.strip() for item in df_Table[2][j].splitlines()]).strip())
                        sqldict['Phone'].append(' '.join([item.strip() for item in df_Table[3][j].splitlines()]).strip())
                        sqldict['BIC SWIFT Code'].append(' '.join([item.strip() for item in df_Table[4][j].splitlines()]).strip())
                        sqldict['RegCtry'].append('UG')
                        sqldict['Cntry'].append('UG')
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['ListCode'].append(d)
                        sqldict['RegCode'].append('BUG')
                        sqldict['RegulationType'].append('Regulated') 
                    sqldict = bourange_same_length_array(sqldict)
            os.remove(filePath)

        elif d ==3:
            prepare_tempfolder(tempfolder)
            existing_files = set(os.listdir(tempfolder))
            driver.find_element(By.XPATH, '//*[@id="apollo-page"]/section/div/div[2]/div[2]/div[1]/div/div/div[2]/div/div/div/ul[1]/li['+ str(d) +']/a').click()
            pdf_file = wait_for_new_download(tempfolder, existing_files)
            filePath = os.path.join(tempfolder, pdf_file)
            
            print(m,'---',typology[m].text)
            tables = pd.read_excel(filePath)

            first_col = tables.columns[1]
            tables = tables[tables[first_col].notna()]
            tables.columns = tables.iloc[0]
            tables = tables[1:]

            second_last_col = tables.columns[-2]
            last_col = tables.columns[-1]

            for _, row in tables.iterrows():
                name = row[second_last_col]
                address = row[last_col]
                # use the two values
                #print(name, address)
                sqldict['Name'].append(name)
                sqldict['Address_1'].append(address)
                sqldict['RegCtry'].append('UG')
                sqldict['Cntry'].append('UG')
                sqldict['ListProcessDate'].append(processdate)
                sqldict['ListCode'].append(d)
                sqldict['RegCode'].append('BUG')
                sqldict['RegulationType'].append('Regulated') 
            sqldict = bourange_same_length_array(sqldict)
            os.remove(filePath)
        elif d ==4:
            prepare_tempfolder(tempfolder)
            existing_files = set(os.listdir(tempfolder))
            driver.find_element(By.XPATH, '//*[@id="apollo-page"]/section/div/div[2]/div[2]/div[1]/div/div/div[2]/div/div/div/ul[1]/li['+ str(d) +']/a').click()
            pdf_file = wait_for_new_download(tempfolder, existing_files)
            filePath = os.path.join(tempfolder, pdf_file)
            
            print(m,'---',typology[m].text)
            tables_4 = pd.read_excel(filePath)

            first_col = tables_4.columns[1]
            tables_4 = tables_4[tables_4[first_col].notna()]
            tables_4.columns = tables_4.iloc[0]
            tables_4 = tables_4[1:]

            second_last_col = tables_4.columns[-2]
            last_col = tables_4.columns[-1]

            for _, row in tables_4.iterrows():
                name = row[second_last_col]
                address = row[last_col]
                # use the two values
                #print(name, address)
                sqldict['Name'].append(name)
                sqldict['Address_1'].append(address)
                sqldict['RegCtry'].append('UG')
                sqldict['Cntry'].append('UG')
                sqldict['ListProcessDate'].append(processdate)
                sqldict['ListCode'].append(d)
                sqldict['RegCode'].append('BUG')
                sqldict['RegulationType'].append('Regulated') 
            sqldict = bourange_same_length_array(sqldict)
            os.remove(filePath)



    
    

Running UG BUG Web Scraping Tool v.1.0
Working with UG BUG 1
0 --- Commercial Banks
Waiting for file to download
['LICENSED-COMMERCIAL-BANKS-AS-AT-3-MARCH-2025.docx']


100%|██████████| 1/1 [00:10<00:00, 10.56s/it]


Waiting for file to download
['LICENSED-CIs-AS-JULY-2025-.pdf']
1 --- Credit Institutions
Waiting for file to download
2 --- Forex Bureaux
Waiting for file to download
3 --- Money Remitters
Waiting for file to download
['LICENSED-MICROFINANCE-DEPOSIT-TAKING-INSTITUTIONS-MDIs.pdf']
4 --- Microfinance Deposit-taking Institutions


In [16]:
df=pd.DataFrame(sqldict)
writer = ExcelWriter(filename)
df.to_excel(writer, 'SQL Ready', index=False)

writer.save()
writer.close()

sleep(3)

driver.quit()

C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_28156\386717110.py:3: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [17]:
df.to_excel(filename)